In [ ]:
import polars as pl

PRICE_PER_RACE = {
    1: 8,
    2: 10,
    3: 12,
    4: 15,
    5: 18,
    6: 18,
    7: 18,
    8: 18,
}

PRICE_PER_RELAY = 12

EVENT_ID = 597

df_individual = pl.read_csv(f"{EVENT_ID}-riepilogo-dbmeeting.csv", separator=";")
# It is dbmeeting style so columns are CodSocietà, Cognome, Nome, Anno, Sesso, Gara1, Gara2, ..., Societa, Regione
N_GARE = (len(df_individual.columns) - 7) // 2  # Subtract non-Gara columns

assert N_GARE == len(PRICE_PER_RACE), "Number of races does not match price mapping."

print(f"Numero di gare per la manifestazione: {N_GARE}")

df_individual.head()

In [ ]:
# Create a DICT with key = CodSocietà and value Societa
individual_grouped = (
    df_individual.select(["CodSocietà", "Societa"])
    .unique()
    .to_dict(as_series=False)
)
SOCIETA_CODE_TO_NAME = dict(zip(individual_grouped["CodSocietà"], individual_grouped["Societa"]))
print(SOCIETA_CODE_TO_NAME)

In [ ]:
# Groupby società and count the number of races for each athlete (Cognome, Nome, Anno, Sesso)
individual_grouped = (
    df_individual.group_by(["CodSocietà", "Cognome", "Nome", "Anno", "Sesso"])
    # Compute the nGare as the count of non-null Gara{i} columns
    .agg(
        pl.sum_horizontal(
            *[pl.col(f"Gara{i}").is_not_null() for i in range(1, N_GARE + 1)]
        ).sum().alias("nGare")
    )
    .with_columns(
        pl.col("nGare").map_elements(lambda x: PRICE_PER_RACE.get(x), return_dtype=pl.UInt32).alias("Totale Atleta"),
        pl.col("CodSocietà").map_elements(lambda x: SOCIETA_CODE_TO_NAME.get(x, x), return_dtype=pl.Utf8).alias("Società")
    )
)
# Reorder columns to swap positions
individual_grouped = individual_grouped.select(["Società", "CodSocietà"] + [col for col in individual_grouped.columns if col not in ["Società", "CodSocietà"]])
individual_grouped.write_csv(f"{EVENT_ID}-riepilogo-pagamenti-dettaglio-individuali.csv", separator=";")
individual_grouped.head()

In [ ]:
# Groupby CodSocietà to compute total price per società
total_individual_df = (
    individual_grouped.group_by("CodSocietà")
    .agg(
        pl.sum("Totale Atleta").alias("Totale Individuali")
    )
)
total_individual_df.head(20)

In [ ]:
df_relay = pl.read_csv(f"{EVENT_ID}-riepilogo-staffette-dbmeeting.csv", separator=";")
df_relay.head()

In [ ]:
total_relay_df = (
    df_relay.group_by("CodSocietà")
    .agg(
        # Just count n rows
        (pl.len() * PRICE_PER_RELAY).alias("Totale Staffette")
    )
)
total_relay_df.head(20)

In [ ]:
# Join total_individual_df and total_relay_df on CodSocietà
final_df = total_individual_df.join(total_relay_df, on="CodSocietà", how="full").fill_null(0)
final_df = final_df.with_columns(
    (pl.col("Totale Individuali") + pl.col("Totale Staffette")).alias("Totale"),
    pl.col("CodSocietà").map_elements(lambda x: SOCIETA_CODE_TO_NAME.get(x, x), return_dtype=pl.Utf8).alias("Società")
).select(["Società", "Totale Individuali", "Totale Staffette", "Totale"]).sort("Totale", descending=True)

final_df = final_df.rename({
    "Totale Individuali": "Totale Individuali (€)",
    "Totale Staffette": "Totale Staffette (€)",
    "Totale": "Totale (€)"
})
final_df.write_csv(f"{EVENT_ID}-riepilogo-pagamenti.csv", separator=";")

final_df.head(20)